In [1]:
import sys
sys.path.append("..")
import pandas as pd
from src.data import load_config, load_contract, time_split, cv_folds, file_hash, ROOT

cfg = load_config()
tf, spec, data = load_contract(cfg)
train, valid = time_split(data, cfg)
print("contract", spec["contract_version"], "| train", len(train), "| valid", len(valid))
print("default rate train / valid:", round(train[cfg["data"]["target"]].mean(), 3),
      "/", round(valid[cfg["data"]["target"]].mean(), 3))

for q, fit_idx, val_idx in cv_folds(train, cfg):
    print(f"fold {q}: train {len(fit_idx):>6,} | validate {len(val_idx):>5,}")

print("data hash:", file_hash(ROOT / cfg["paths"]["contract_dir"] / "analytical_dataset.parquet")[:16], "...")

contract 1.0.0 | train 14373 | valid 15627
default rate train / valid: 0.172 / 0.201
fold 2014Q1: train  5,349 | validate 1,906
fold 2014Q2: train  7,255 | validate 2,091
fold 2014Q3: train  9,346 | validate 2,336
fold 2014Q4: train 11,682 | validate 2,691
data hash: 9cc2f55230d13567 ...


In [2]:
from src.metrics import ranking_metrics

TARGET = cfg["data"]["target"]
GRADE_NUM = {g: i for i, g in enumerate("ABCDEFG", start=1)}

def evaluate_score(name, score_fn):
    """Score each CV validation quarter and 2015 with score_fn(df) -> risk score; return one row per period."""
    rows = []
    for q, _, val_idx in cv_folds(train, cfg):
        v = train.loc[val_idx]
        rows.append({"model": name, "period": q, **ranking_metrics(v[TARGET], score_fn(v))})
    rows.append({"model": name, "period": "2015", **ranking_metrics(valid[TARGET], score_fn(valid))})
    return pd.DataFrame(rows)

grade_results = evaluate_score("grade alone", lambda df: df["grade"].map(GRADE_NUM))
display(grade_results.round(3))
cv = grade_results[grade_results["period"] != "2015"]
print("CV mean AUC:", round(cv["auc"].mean(), 3), "| 2015 AUC:", round(grade_results.iloc[-1]["auc"], 3))

,model,period,auc,gini,ks
0,grade alone,2014Q1,0.681,0.362,0.281
1,grade alone,2014Q2,0.670,0.340,0.247
2,grade alone,2014Q3,0.688,0.375,0.257
3,grade alone,2014Q4,0.714,0.427,0.310
4,grade alone,2015,0.703,0.405,0.299


CV mean AUC: 0.688 | 2015 AUC: 0.703


In [3]:
from src.features import MODEL_FEATURES

iv = pd.Series({c: spec["woe_bins"][c]["iv"] for c in MODEL_FEATURES}).sort_values(ascending=False)
woe_train = tf.woe(train)                                     # WoE values from the contract's bins

# Rule 1: IV >= 0.02
rule1 = iv[iv >= 0.02].index.tolist()
print("dropped, IV < 0.02:", {c: round(iv[c], 4) for c in iv.index if c not in rule1})

# Rule 2: greedy by IV, |WoE correlation| <= 0.7 with everything already kept
corr = woe_train[[f"woe_{c}" for c in rule1]].corr().abs()
selected, dropped_corr = [], {}
for c in rule1:                                               # already ordered from highest IV
    clash = [k for k in selected if corr.loc[f"woe_{c}", f"woe_{k}"] > 0.7]
    if clash:
        dropped_corr[c] = f"corr {corr.loc[f'woe_{c}', f'woe_{clash[0]}']:.2f} with {clash[0]}"
    else:
        selected.append(c)
print("dropped, correlated:", dropped_corr)
print(f"\nselected ({len(selected)}):")
print(iv[selected].round(3).to_string())

dropped, IV < 0.02: {'loan_amnt': np.float64(0.0193), 'mort_acc': np.float64(0.0192), 'credit_hist_months': np.float64(0.0115), 'open_acc': np.float64(0.0094), 'revol_bal_log': np.float64(0.0024), 'total_acc': np.float64(0.0023), 'mths_since_last_delinq': np.float64(0.0017), 'ever_delinq': np.float64(0.0014), 'pub_rec': np.float64(0.0009), 'pub_rec_bankruptcies': np.float64(0.0009), 'bc_util_missing': np.float64(0.0008), 'delinq_2yrs': np.float64(0.0)}
dropped, correlated: {'tot_hi_cred_lim_log': 'corr 0.83 with avg_cur_bal_log', 'tot_cur_bal_log': 'corr 0.81 with avg_cur_bal_log', 'revol_util_cap': 'corr 0.76 with bc_util_cap'}

selected (14):
term_months             0.195
fico_mid                0.099
loan_to_income          0.098
acc_open_past_24mths    0.075
dti_cap                 0.062
num_tl_op_past_12m      0.049
verification_grp        0.046
avg_cur_bal_log         0.041
bc_util_cap             0.036
annual_inc_log          0.035
inq_last_6mths          0.033
home_ownership   

In [4]:
from sklearn.linear_model import LogisticRegression

woe_all = tf.woe(data)                       # WoE for every loan (same index as data)
y_all = data[TARGET]

def cv_evaluate(cols, make_model, X=woe_all):
    """Fit on each fold's training part, score its validation quarter; return per-fold metrics + out-of-fold PDs."""
    rows, oof = [], pd.Series(index=train.index, dtype=float)
    for q, fit_idx, val_idx in cv_folds(train, cfg):
        model = make_model().fit(X.loc[fit_idx, cols], y_all.loc[fit_idx])
        p = model.predict_proba(X.loc[val_idx, cols])[:, 1]
        oof.loc[val_idx] = p
        rows.append({"period": q, **ranking_metrics(y_all.loc[val_idx], p)})
    return pd.DataFrame(rows), oof

sc_cols = [f"woe_{c}" for c in selected]
sc_cv, sc_oof = cv_evaluate(sc_cols, lambda: LogisticRegression(max_iter=1000))
display(sc_cv.round(3))
print("scorecard CV mean AUC:", round(sc_cv["auc"].mean(), 3), "| grade CV mean AUC: 0.688")

final_sc = LogisticRegression(max_iter=1000).fit(woe_all.loc[train.index, sc_cols], y_all.loc[train.index])
weights = pd.Series(final_sc.coef_[0], index=selected).round(3).sort_values()
print(weights.to_string())
print("rule 3, all weights negative:", bool((weights < 0).all()))

,period,auc,gini,ks
0,2014Q1,0.700,0.400,0.324
1,2014Q2,0.678,0.355,0.257
2,2014Q3,0.716,0.432,0.335
3,2014Q4,0.718,0.435,0.337


scorecard CV mean AUC: 0.703 | grade CV mean AUC: 0.688
term_months            -0.944
purpose_grp            -0.813
inq_last_6mths         -0.757
acc_open_past_24mths   -0.713
home_ownership         -0.696
bc_util_cap            -0.637
fico_mid               -0.627
annual_inc_log         -0.627
total_rev_hi_lim_log   -0.502
avg_cur_bal_log        -0.479
loan_to_income         -0.439
dti_cap                -0.408
num_tl_op_past_12m     -0.364
verification_grp       -0.281
rule 3, all weights negative: True


In [5]:
variants = []

for C in [0.01, 0.1, 1.0, 10.0]:                                    # 1. regularization
    res, _ = cv_evaluate(sc_cols, lambda C=C: LogisticRegression(C=C, max_iter=1000))
    variants.append({"variant": f"C = {C}", "cv_auc": res["auc"].mean()})

corr_all = woe_all.loc[train.index].corr().abs()
for extra in [["loan_amnt"], ["mort_acc"], ["loan_amnt", "mort_acc"]]:   # 2. borderline features
    cols = sc_cols + [f"woe_{c}" for c in extra]
    res, _ = cv_evaluate(cols, lambda: LogisticRegression(max_iter=1000))
    m = LogisticRegression(max_iter=1000).fit(woe_all.loc[train.index, cols], y_all.loc[train.index])
    max_corr = max(corr_all.loc[f"woe_{c}", sc_cols].max() for c in extra)
    variants.append({"variant": "+ " + " + ".join(extra), "cv_auc": res["auc"].mean(),
                     "all_negative": bool((m.coef_[0] < 0).all()), "max_corr_with_selected": round(max_corr, 2)})

display(pd.DataFrame(variants).round(4))

,variant,cv_auc,all_negative,max_corr_with_selected
0,C = 0.01,0.6993,NaN,NaN
1,C = 0.1,0.7030,NaN,NaN
2,C = 1.0,0.7028,NaN,NaN
3,C = 10.0,0.7027,NaN,NaN
4,+ loan_amnt,0.7024,False,0.57
5,+ mort_acc,0.7028,True,0.52
6,+ loan_amnt + mort_acc,0.7024,True,0.57


In [6]:
from src.experiments import log_experiment

DATA_HASH = file_hash(ROOT / cfg["paths"]["contract_dir"] / "analytical_dataset.parquet")
log = lambda *a, **k: log_experiment(cfg, *a, data_hash=DATA_HASH, contract_version=spec["contract_version"], **k)

grade_cv = grade_results[grade_results["period"] != "2015"]
log("grade_alone", "benchmark", grade_cv, ["grade"], {}, notes="no training; A=1..G=7")
log("scorecard", "base: IV>=0.02, corr<=0.7, C=1", sc_cv, selected, {"C": 1.0},
    notes="all weights negative; chosen")

for C in [0.01, 0.1, 10.0]:
    res, _ = cv_evaluate(sc_cols, lambda C=C: LogisticRegression(C=C, max_iter=1000))
    log("scorecard", f"C={C}", res, selected, {"C": C}, notes="no gain >= 0.002 vs C=1")
for extra, note in [(["loan_amnt"], "weight turned positive (overlaps loan_to_income)"),
                    (["mort_acc"], "no gain"), (["loan_amnt", "mort_acc"], "no gain")]:
    res, _ = cv_evaluate(sc_cols + [f"woe_{c}" for c in extra], lambda: LogisticRegression(max_iter=1000))
    log("scorecard", "+ " + " + ".join(extra), res, selected + extra, {"C": 1.0}, notes=note)

display(pd.read_csv(ROOT / cfg["paths"]["experiment_log"])[["model", "variant", "cv_auc_mean", "cv_ks_mean", "notes"]])

,model,variant,cv_auc_mean,cv_ks_mean,notes
0,grade_alone,benchmark,0.6881,0.2737,no training; A=1..G=7
1,scorecard,"base: IV>=0.02, corr<=0.7, C=1",0.7028,0.3133,all weights negative; chosen
2,scorecard,C=0.01,0.6993,0.3057,no gain >= 0.002 vs C=1
3,scorecard,C=0.1,0.7030,0.3095,no gain >= 0.002 vs C=1
4,scorecard,C=10.0,0.7027,0.3143,no gain >= 0.002 vs C=1
5,scorecard,+ loan_amnt,0.7024,0.3126,weight turned positive (overlaps loan_to_income)
6,scorecard,+ mort_acc,0.7028,0.3130,no gain
7,scorecard,+ loan_amnt + mort_acc,0.7024,0.3114,no gain
